# FedPartSAM — Experiment 1 Smoke Test on Google Colab

Verifies that **all 8 strategies** of Experiment 1 run end-to-end on a T4 GPU:

`fedavg`, `fedprox`, `fedadam`, `fedpartavg`, `fedpartadam`, `fedpartsam_m`, `fedpartsam_d`, `fedpseudogradsim`

**Setup**: ResNet-18 (GroupNorm) + CIFAR-100, **5 clients** (reduced from the paper's 10 to fit Colab free-tier's 12.7 GB RAM — see note below), IID, single seed, 10 rounds total (2 warmup + 8 layer-wise with `rounds_per_layer=2`, so 4 layers exercised), 2 local epochs each.

Results are saved to your Google Drive after each completed strategy, so a disconnect won't lose progress — just re-run the smoke cell to resume.

> **Why 5 clients on Colab?** With 10 Ray client actors + a ResNet-18 model + the CIFAR-100 dataset partition + the Python driver, Colab free-tier's 12.7 GB RAM gets saturated and Ray's memory monitor starts killing workers mid-fit. Once a worker is killed, Flower's aggregator receives no client updates and the global model stops improving — symptom: identical loss/accuracy for every round. Reducing `--num-clients` to 5 halves the worker count and the per-client data size. For the paper's real 10-client runs you'd need a Colab Pro high-RAM runtime or a local machine with more memory.

## 1. Pick GPU runtime

Before running anything: **Runtime → Change runtime type → T4 GPU → Save**.

Then run the cell below to confirm CUDA is visible.

In [ ]:
import torch
assert torch.cuda.is_available(), "GPU not available — set Runtime → Change runtime type → T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)

## 2. Clone the repo and install Flower

Colab already has `torch`, `numpy`, `matplotlib`, `pandas` — we only need to add Flower.

In [ ]:
!git clone https://github.com/ViragSimon/layer_momentum.git
%cd layer_momentum
!pip install -q flwr==1.15.1 flwr-datasets==0.5.0 ray==2.31.0

## 3. Mount Google Drive (so results survive disconnects)

Pickle files are written here after every completed run — you can re-open the notebook and pick up where you left off.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

RESULTS_DIR = '/content/drive/MyDrive/layer_momentum_smoke'
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Results will be saved to:', RESULTS_DIR)

## 4. Run the smoke test

**What this does**: runs all 8 strategies in Experiment 1 once each (single seed, `seed=42`, 5 clients).

Layer-wise schedule for the partial strategies:

| Round | Active layer |
|---|---|
| 1, 2 | -1 (warmup, full model) |
| 3, 4 | layer 0 — `conv1 + gn1` |
| 5, 6 | layer 1 — `layer1` (BasicBlock group) |
| 7, 8 | layer 2 — `layer2` |
| 9, 10 | layer 3 — `layer3` |

This covers 4 of the 6 ResNet-18 logical layer groups (`layer4` and `fc` aren't reached). To cover **all 6 layers**, change `--rounds 10` to `--rounds 14` below.

**Resumable**: each completed strategy's result is appended to `experiment_1.pkl` immediately. If Colab disconnects mid-run, just re-run this cell — it skips strategies already in the pickle and continues from the next one.

**If you see identical loss/accuracy for every round** (e.g. `acc=0.0100` over and over), that means Ray is killing workers mid-fit due to OOM. Stop the cell, delete the corrupted pickle with `!rm {RESULTS_DIR}/experiment_1.pkl`, and re-run with a smaller `--num-clients` (or upgrade to a high-RAM Colab runtime).

**Expected wall time on T4**: ~60–90 minutes for all 8 strategies with `--num-clients 5`.

In [ ]:
!python experiments/run_all.py \
    --experiment 1 \
    --seeds 42 \
    --rounds 10 \
    --warmup-rounds 2 \
    --rounds-per-layer 2 \
    --local-epochs 2 \
    --num-clients 5 \
    --output-dir {RESULTS_DIR}

## 5. Inspect results

Loads the saved pickle and prints a summary table — final accuracy, total bytes transmitted, and wall time per strategy.

In [ ]:
import pickle

with open(f'{RESULTS_DIR}/experiment_1.pkl', 'rb') as f:
    runs = pickle.load(f)

print(f'Total runs: {len(runs)}\n')
print(f'{"strategy":<22s}  {"final_acc":>10s}  {"total_MB":>10s}  {"time_min":>10s}')
print('-' * 60)
for r in runs:
    cfg = r['config']
    eval_results = r['results']['eval_results']
    round_results = r['results']['round_results']
    last_round = max(eval_results)
    final_acc = eval_results[last_round]['accuracy']
    total_bytes = sum(rr.get('total_bytes', 0) for rr in round_results.values())
    total_mb = total_bytes / (1024 ** 2)
    elapsed_min = r['elapsed_seconds'] / 60
    print(f'{cfg["strategy"]:<22s}  {final_acc:>10.4f}  {total_mb:>10.2f}  {elapsed_min:>10.1f}')

## 6. Verify partial strategies cycled through layers

Sanity check that the layer-wise schedule actually fired for the partial strategies. Should print `[-1, -1, 0, 0, 1, 1, 2, 2, 3, 3]` for any `fedpart*` / `fedpartsam_*` / `fedpseudogradsim` run.

In [ ]:
for r in runs:
    name = r['config']['strategy']
    if not name.startswith('fedpart') and name != 'fedpseudogradsim':
        continue
    rounds = sorted(r['results']['round_results'].keys())
    layers = [r['results']['round_results'][rd].get('active_layer', '?') for rd in rounds]
    print(f'{name:<22s} {layers}')

## 7. (Optional) Quick accuracy plot

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))
for r in runs:
    name = r['config']['strategy']
    eval_results = r['results']['eval_results']
    rounds = sorted(eval_results.keys())
    accs = [eval_results[rd]['accuracy'] for rd in rounds]
    ax.plot(rounds, accs, marker='o', label=name)
ax.set_xlabel('Round')
ax.set_ylabel('Test accuracy')
ax.set_title('Experiment 1 smoke — CIFAR-100, ResNet-18, 1 seed')
ax.legend(fontsize=8, loc='best')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()